In [1]:
%load_ext dotenv
%dotenv /home/aurora/.env
%matplotlib inline

In [2]:
import numpy as np
import pandas as pd
import datetime as dt
from google.cloud import bigquery
import re
import pymongo
import os
from flatten_json import flatten

In [3]:
# Automated date and time
start = dt.datetime(2019,5,28)
start = start.replace(hour=0, minute=0, second=0, microsecond=0)
#start = start - dt.timedelta(hours = 5, minutes = 30)
end = dt.datetime(2019,5,31)
end = end.replace(hour=0, minute=0, second=0, microsecond=0)
#end = end - dt.timedelta(hours = 5, minutes = 30)
print(start,end,end-start)

2019-05-28 00:00:00 2019-05-31 00:00:00 3 days, 0:00:00


In [4]:
client = bigquery.Client.from_service_account_json('/home/analytics/.secure_files/hitwicketsuperstars-f3e8c620a88c.json')
query = (
    f"""SELECT
    * FROM 
    `hitwicketsuperstars.analytics_190927423.new_user_reference`"""
)
new_user_reference = client.query(query).to_dataframe()

In [5]:
new_user_reference['user_first_touch_timestamp'] = new_user_reference['user_first_touch_timestamp'].astype('datetime64[s]')
android_new = new_user_reference[(new_user_reference['user_first_touch_timestamp'] >= start) & 
                                 (new_user_reference['user_first_touch_timestamp'] < end) & 
                                 (new_user_reference['platform'] == 'ANDROID')]

In [6]:
android_new['string_value']= 'opened_app'

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: http://pandas.pydata.org/pandas-docs/stable/indexing.html#indexing-view-versus-copy
  """Entry point for launching an IPython kernel.


In [7]:
android_new.head()

,device_id,user_first_touch_timestamp,platform,string_value
27,24d4bf8c69096ccd898395a972995564,2019-05-30 00:46:56,ANDROID,opened_app
28,42423648c200e2c3b0f99b33073dee1d,2019-05-30 17:50:56,ANDROID,opened_app
29,ea643545b819aee479280c2aeb6af4cf,2019-05-30 11:52:32,ANDROID,opened_app
30,b2c5c8f1133f130b39cd75adf1b2109b,2019-05-30 13:00:48,ANDROID,opened_app
31,8ac69bb2005ec9ea10509a5680473ce2,2019-05-30 05:20:00,ANDROID,opened_app


In [8]:
len(android_new)

2613

In [9]:
query = (
    f"""SELECT
    user_id as device_id,
  device.mobile_os_hardware_model as device
FROM `hitwicketsuperstars.analytics_190927423.events_`
      WHERE _TABLE_SUFFIX BETWEEN '0528'
      AND '0531'
      GROUP BY user_id, device
      """
)
df_all = client.query(query).to_dataframe()

In [10]:
df_all.head()

,device_id,device
0,696990eb87f3143da1e217fef697b540,Moto G (5)
1,82923b6025bcd2101f7d8b2054282bcb,Redmi 4A
2,2867da3fe7e535ae4d966601e1974a6c,RMX1825
3,None,SM-J200G
4,96d1f1c4e2e10c341e10b20333a399fb,ONEPLUS A3003


In [11]:
android_new1 = pd.merge(android_new,df_all,on='device_id')

In [12]:
android_new1.head()

,device_id,user_first_touch_timestamp,platform,string_value,device
0,24d4bf8c69096ccd898395a972995564,2019-05-30 00:46:56,ANDROID,opened_app,A1601
1,42423648c200e2c3b0f99b33073dee1d,2019-05-30 17:50:56,ANDROID,opened_app,JSN-L42
2,ea643545b819aee479280c2aeb6af4cf,2019-05-30 11:52:32,ANDROID,opened_app,LLD-AL20
3,b2c5c8f1133f130b39cd75adf1b2109b,2019-05-30 13:00:48,ANDROID,opened_app,Redmi Note 5 Pro
4,8ac69bb2005ec9ea10509a5680473ce2,2019-05-30 05:20:00,ANDROID,opened_app,Redmi Note 6 Pro


In [14]:
len(android_new1)

2613

In [18]:
ab = android_new1.groupby('device')['device_id'].count().reset_index()

In [20]:
ab.sort_values('device_id',ascending=False,inplace=True)

In [25]:
ab.head(25)

,device,device_id
312,Redmi 5A,65
319,Redmi Note 4,64
392,SM-J200G,63
522,vivo 1606,55
321,Redmi Note 5 Pro,53
65,CPH1803,50
393,SM-J210F,47
315,Redmi 6A,46
414,SM-J701F,43
365,SM-G610F,40


In [26]:
low_phones = ['Redmi 5A','Redmi 4A','Redmi 6A','SM-J250F','vivo 1606','A37f','SM-J200G','CPH1803','SM-J210F','SM-J260G']

In [30]:
high_phones = ['Mi A1','ONEPLUS A6000','vivo 1610','Redmi Note 5 Pro','Redmi Note 4','SM-J701F','A37fw','Redmi Note 3','Redmi 4','SM-J600G','SM-G610F','SM-J600G','SM-G615F',
               'A1601','POCO F1','Redmi Y2']

In [31]:
android_funnel_low = android_new1[android_new1["device"].isin(low_phones)]

In [32]:
len(android_funnel_low)

424

In [33]:
android_funnel_high = android_new1[android_new1["device"].isin(high_phones)]

In [34]:
len(android_funnel_low) , len(android_funnel_high)

(424, 438)

In [35]:
android_funnel_low.columns = ['device_id','first_touch','platform','Label','device']
android_funnel_high.columns = ['device_id','first_touch','platform','Label','device']

#table_low = android_funnel_low.pivot_table(index = 'Label', columns = 'device', values = 'device_id',aggfunc = 'count')
#table_high= android_funnel_high.pivot_table(index = 'Label', columns = 'device', values = 'device_id',aggfunc = 'count')
table_low = android_funnel_low.groupby('Label')['device_id'].count().reset_index()
table_low.set_index('Label',inplace=True)
table_high = android_funnel_high.groupby('Label')['device_id'].count().reset_index()
table_high.set_index('Label',inplace=True)

In [36]:
table_low

,device_id
Label,
opened_app,424


In [37]:
table_high

,device_id
Label,
opened_app,438


In [38]:
query = (
    f"""SELECT
  user_id,
  event_name,
  event_timestamp,
  params.key,
  params.value.string_value,
  platform,
  device.mobile_brand_name
FROM `hitwicketsuperstars.analytics_190927423.events_2019*`, 
  UNNEST(event_params) AS params
  WHERE event_name IN ('ftue','signup')
  AND _TABLE_SUFFIX BETWEEN '0528'
  AND '0531'"""
)
df = client.query(query).to_dataframe()
df.head()

,user_id,event_name,event_timestamp,key,string_value,platform,mobile_os_hardware_model
0,b36db154f41506e7d24a62697b3c9922,ftue,1559050443394006,action,natasha_next_opponent_tough_seen,ANDROID,K016
1,b36db154f41506e7d24a62697b3c9922,ftue,1559050443394006,firebase_event_origin,app,ANDROID,K016
2,b36db154f41506e7d24a62697b3c9922,ftue,1559050443394006,ga_session_number,None,ANDROID,K016
3,b36db154f41506e7d24a62697b3c9922,ftue,1559050443394006,firebase_screen_id,None,ANDROID,K016
4,b36db154f41506e7d24a62697b3c9922,ftue,1559050443394006,firebase_screen,rivals_page,ANDROID,K016


In [39]:
original_df = df[df['key']=='action']

In [40]:
df = original_df.copy()

In [41]:
df.head()

,user_id,event_name,event_timestamp,key,string_value,platform,mobile_os_hardware_model
0,b36db154f41506e7d24a62697b3c9922,ftue,1559050443394006,action,natasha_next_opponent_tough_seen,ANDROID,K016
7,b36db154f41506e7d24a62697b3c9922,ftue,1559050443915007,action,natasha_next_opponent_tough_clicked,ANDROID,K016
14,b36db154f41506e7d24a62697b3c9922,ftue,1559051029359005,action,training_center_crucial_seen,ANDROID,K016
21,b36db154f41506e7d24a62697b3c9922,ftue,1559051030840006,action,training_center_crucial_clicked,ANDROID,K016
28,b36db154f41506e7d24a62697b3c9922,ftue,1559051030841007,action,hand_pointer_train_center_seen,ANDROID,K016


In [42]:
len(df)

240537

In [43]:
df = pd.merge(df,android_new1,left_on='user_id',right_on='device_id')

In [44]:
df.head()

,user_id,event_name,event_timestamp,key,string_value_x,platform_x,mobile_os_hardware_model,device_id,user_first_touch_timestamp,platform_y,string_value_y,device
0,d12b659430ec95d63bf2505cad722941,ftue,1559059349809001,action,natasha_stadium_ready_inaugrate_seen,ANDROID,Redmi 4,d12b659430ec95d63bf2505cad722941,2019-05-28 10:09:57,ANDROID,opened_app,Redmi 4
1,d12b659430ec95d63bf2505cad722941,ftue,1559059379995002,action,natasha_stadium_ready_inaugrate_seen,ANDROID,Redmi 4,d12b659430ec95d63bf2505cad722941,2019-05-28 10:09:57,ANDROID,opened_app,Redmi 4
2,d12b659430ec95d63bf2505cad722941,ftue,1559059381446003,action,natasha_stadium_ready_inaugrate_clicked,ANDROID,Redmi 4,d12b659430ec95d63bf2505cad722941,2019-05-28 10:09:57,ANDROID,opened_app,Redmi 4
3,d12b659430ec95d63bf2505cad722941,ftue,1559059381448004,action,hand_pointer_inaugrate_seen,ANDROID,Redmi 4,d12b659430ec95d63bf2505cad722941,2019-05-28 10:09:57,ANDROID,opened_app,Redmi 4
4,d12b659430ec95d63bf2505cad722941,ftue,1559147506212004,action,training_center_crucial_seen,ANDROID,Redmi 4,d12b659430ec95d63bf2505cad722941,2019-05-28 10:09:57,ANDROID,opened_app,Redmi 4


In [45]:
df.drop(['event_name','key','string_value_y','device_id','platform_y','mobile_os_hardware_model'], axis=1, inplace=True)

In [46]:
df.head()

,user_id,event_timestamp,string_value_x,platform_x,user_first_touch_timestamp,device
0,d12b659430ec95d63bf2505cad722941,1559059349809001,natasha_stadium_ready_inaugrate_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
1,d12b659430ec95d63bf2505cad722941,1559059379995002,natasha_stadium_ready_inaugrate_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
2,d12b659430ec95d63bf2505cad722941,1559059381446003,natasha_stadium_ready_inaugrate_clicked,ANDROID,2019-05-28 10:09:57,Redmi 4
3,d12b659430ec95d63bf2505cad722941,1559059381448004,hand_pointer_inaugrate_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
4,d12b659430ec95d63bf2505cad722941,1559147506212004,training_center_crucial_seen,ANDROID,2019-05-28 10:09:57,Redmi 4


In [47]:
df.columns = ['user_id','event_timestamp','string_value','platform','user_firt_touch_timestamp','device']

In [48]:
df.head()

,user_id,event_timestamp,string_value,platform,user_firt_touch_timestamp,device
0,d12b659430ec95d63bf2505cad722941,1559059349809001,natasha_stadium_ready_inaugrate_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
1,d12b659430ec95d63bf2505cad722941,1559059379995002,natasha_stadium_ready_inaugrate_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
2,d12b659430ec95d63bf2505cad722941,1559059381446003,natasha_stadium_ready_inaugrate_clicked,ANDROID,2019-05-28 10:09:57,Redmi 4
3,d12b659430ec95d63bf2505cad722941,1559059381448004,hand_pointer_inaugrate_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
4,d12b659430ec95d63bf2505cad722941,1559147506212004,training_center_crucial_seen,ANDROID,2019-05-28 10:09:57,Redmi 4


In [49]:
len(df[df['device'].isin(high_phones)])

33994

In [50]:
map1 = pd.read_csv("/home/aurora/analytics/scratchpad/Maps/mod_map1.csv")
map1.head()

,Label,Action,Order
0,Opened App,opened_app,0
1,New User Hand Pointer Clicked,new_user_hand_pointer_click,1
2,First Welcome Natasha Seen,natasha_welcome1_seen,2
3,First Welcome Natasha Clicked,natasha_welcome1_clicked,3
4,Second Welcome Natasha Seen,natasha_welcome2_seen,4


In [51]:
user_funnel = df[df["string_value"].isin(map1["Action"])]

In [52]:
len(user_funnel)

183726

In [53]:
user_funnel.head()

,user_id,event_timestamp,string_value,platform,user_firt_touch_timestamp,device
23,d12b659430ec95d63bf2505cad722941,1559038200898002,new_user_hand_pointer_click,ANDROID,2019-05-28 10:09:57,Redmi 4
24,d12b659430ec95d63bf2505cad722941,1559038223464001,natasha_welcome1_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
25,d12b659430ec95d63bf2505cad722941,1559038227075000,natasha_welcome1_clicked,ANDROID,2019-05-28 10:09:57,Redmi 4
26,d12b659430ec95d63bf2505cad722941,1559038227090001,natasha_welcome2_seen,ANDROID,2019-05-28 10:09:57,Redmi 4
27,d12b659430ec95d63bf2505cad722941,1559038234096002,natasha_welcome2_clicked,ANDROID,2019-05-28 10:09:57,Redmi 4


In [54]:
len(user_funnel[user_funnel['device'].isin(high_phones)])

31469

In [55]:
user_funnel = user_funnel.sort_values(by = ["user_id","event_timestamp"])
user_funnel.drop_duplicates(subset = ['user_id','string_value'], inplace = True)

In [56]:
len(user_funnel)

180388

In [57]:
len(user_funnel[user_funnel['device'].isin(high_phones)])

30948

In [58]:
high_phones

['Mi A1',
 'ONEPLUS A6000',
 'vivo 1610',
 'Redmi Note 5 Pro',
 'Redmi Note 4',
 'SM-J701F',
 'A37fw',
 'Redmi Note 3',
 'Redmi 4',
 'SM-J600G',
 'SM-G610F',
 'SM-J600G',
 'SM-G615F',
 'A1601',
 'POCO F1',
 'Redmi Y2']

In [59]:
user_funnel_low = user_funnel[user_funnel["device"].isin(low_phones)]

In [60]:
user_funnel_low.head()

,user_id,event_timestamp,string_value,platform,user_firt_touch_timestamp,device
66113,00158f79bb2283d98e16188a77890821,1559187956505001,new_user_hand_pointer_click,ANDROID,2019-05-30 03:45:53,Redmi 5A
66114,00158f79bb2283d98e16188a77890821,1559187987958001,natasha_welcome1_seen,ANDROID,2019-05-30 03:45:53,Redmi 5A
66117,00158f79bb2283d98e16188a77890821,1559187992073000,natasha_welcome1_clicked,ANDROID,2019-05-30 03:45:53,Redmi 5A
66118,00158f79bb2283d98e16188a77890821,1559187992090001,natasha_welcome2_seen,ANDROID,2019-05-30 03:45:53,Redmi 5A
66119,00158f79bb2283d98e16188a77890821,1559187993169002,natasha_welcome2_clicked,ANDROID,2019-05-30 03:45:53,Redmi 5A


In [61]:
user_funnel_high = user_funnel[user_funnel["device"].isin(high_phones)]

In [62]:
user_funnel_high.head()

,user_id,event_timestamp,string_value,platform,user_firt_touch_timestamp,device
174422,011fc8927f0d4662077cc1d2e3020f2d,1559204877406002,new_user_hand_pointer_click,ANDROID,2019-05-30 08:27:55,Redmi Note 5 Pro
174423,011fc8927f0d4662077cc1d2e3020f2d,1559204889064000,natasha_welcome1_seen,ANDROID,2019-05-30 08:27:55,Redmi Note 5 Pro
174424,011fc8927f0d4662077cc1d2e3020f2d,1559204891236000,natasha_welcome1_clicked,ANDROID,2019-05-30 08:27:55,Redmi Note 5 Pro
174425,011fc8927f0d4662077cc1d2e3020f2d,1559204891236001,natasha_welcome2_seen,ANDROID,2019-05-30 08:27:55,Redmi Note 5 Pro
174426,011fc8927f0d4662077cc1d2e3020f2d,1559204892697002,natasha_welcome2_clicked,ANDROID,2019-05-30 08:27:55,Redmi Note 5 Pro


In [63]:
#table_a_low = user_funnel_low.pivot_table(index = 'string_value', columns = 'device', values = 'user_id',aggfunc = 'count')
table_a_low = user_funnel_low.groupby('string_value')['user_id'].count().reset_index()
table_a_low.columns = ['string_value','device_id']
table_a_high = user_funnel_high.groupby('string_value')['user_id'].count().reset_index()
table_a_high.columns = ['string_value','device_id']

In [64]:
table_a_low.head()

,string_value,device_id
0,crete_team_next_clicked,130
1,hand_pointer_dash_scout_01clicked,377
2,hand_pointer_dash_scout_01seen,376
3,hand_pointer_dash_scout_11clicked,171
4,hand_pointer_dash_scout_11seen,173


In [65]:
table_a_high.head()

,string_value,device_id
0,crete_team_next_clicked,152
1,hand_pointer_dash_scout_01clicked,404
2,hand_pointer_dash_scout_01seen,407
3,hand_pointer_dash_scout_11clicked,191
4,hand_pointer_dash_scout_11seen,196


In [66]:
table1_low = pd.merge(map1,table_a_low,right_on='string_value',left_on='Action')

In [67]:
table1_high = pd.merge(map1,table_a_high,right_on='string_value',left_on='Action')

In [68]:
table1_low.sort_values('Order',inplace=True)
table1_high.sort_values('Order',inplace=True)

In [69]:
table1_low.drop(['Action','Order','string_value'], axis=1, inplace=True)
table1_high.drop(['Action','Order','string_value'], axis=1, inplace=True)

In [70]:
table1_low.set_index('Label',inplace=True)
table1_high.set_index('Label',inplace=True)

In [71]:
table1_low.head()

,device_id
Label,
New User Hand Pointer Clicked,402
First Welcome Natasha Seen,391
First Welcome Natasha Clicked,385
Second Welcome Natasha Seen,385
Second Welcome Natasha Clicked,383


In [72]:
table1_high.head()

,device_id
Label,
New User Hand Pointer Clicked,425
First Welcome Natasha Seen,418
First Welcome Natasha Clicked,416
Second Welcome Natasha Seen,416
Second Welcome Natasha Clicked,414


In [73]:
final_table_low = pd.concat([table_low, table1_low])
final_table_high = pd.concat([table_high, table1_high])

In [74]:
final_table_low.head()

,device_id
Label,
opened_app,424
New User Hand Pointer Clicked,402
First Welcome Natasha Seen,391
First Welcome Natasha Clicked,385
Second Welcome Natasha Seen,385


In [75]:
final_table_high.head()

,device_id
Label,
opened_app,438
New User Hand Pointer Clicked,425
First Welcome Natasha Seen,418
First Welcome Natasha Clicked,416
Second Welcome Natasha Seen,416


In [76]:
final_table = pd.merge(final_table_low,final_table_high,left_index=True,right_index=True)

In [77]:
final_table.columns = ['lower_end_devices_users','higher_end_device_users']

In [78]:
final_table.head()

,lower_end_devices_users,higher_end_device_users
Label,,
opened_app,424,438
New User Hand Pointer Clicked,402,425
First Welcome Natasha Seen,391,418
First Welcome Natasha Clicked,385,416
Second Welcome Natasha Seen,385,416


In [79]:
final_table.to_csv('higher_vs_lower_end_comparison_31th_may.csv')

In [80]:
table_percentage = (final_table.divide(final_table.ix[0] / 100)).round()

/home/aurora/miniconda3/lib/python3.7/site-packages/ipykernel_launcher.py:1: DeprecationWarning: 
.ix is deprecated. Please use
.loc for label based indexing or
.iloc for positional indexing

See the documentation here:
http://pandas.pydata.org/pandas-docs/stable/indexing.html#ix-indexer-is-deprecated
  """Entry point for launching an IPython kernel.


In [81]:
table_percentage.head()

,lower_end_devices_users,higher_end_device_users
Label,,
opened_app,100.0,100.0
New User Hand Pointer Clicked,95.0,97.0
First Welcome Natasha Seen,92.0,95.0
First Welcome Natasha Clicked,91.0,95.0
Second Welcome Natasha Seen,91.0,95.0


In [82]:
table_percentage.to_csv('higher_end_comparison_31th_may_percentage_decrease.csv')